# Section 6 — Capstone: Production-Grade Scraper

### **Project Overview**

**Target Website**

We’ll use a real, stable public API-backed site:

    https://quotes.toscrape.com

Why this site:

- designed for scraping
- has pagination
- clean structure
- realistic enough

---

### **What the Scraper Will Do**

The scraper will:

1. scrape quotes from multiple pages
2. extract:
    - quote text
    - author
    - tags
3. handle pagination
4. apply delays (throttling)
5. retry on failure (backoff)
6. store results

---

### **Step 1 — Basic Scraper (Foundation)**

In [3]:
!pip install beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\affine\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://quotes.toscrape.com/page/1/"

response = requests.get(url)

soup = BeautifulSoup(response.text, "html.parser")

quotes = soup.find_all("div", class_="quote")

for q in quotes:
    text = q.find("span", class_="text").text
    author = q.find("small", class_="author").text

    print(text, "-", author)

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” - Albert Einstein
“It is our choices, Harry, that show what we truly are, far more than our abilities.” - J.K. Rowling
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” - Albert Einstein
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” - Jane Austen
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.” - Marilyn Monroe
“Try not to become a man of success. Rather become a man of value.” - Albert Einstein
“It is better to be hated for what you are than to be loved for what you are not.” - André Gide
“I have not failed. I've just found 10,000 ways that won't work.” - Thomas A. Edison
“A woman is like a tea bag; you never know how strong it is until it's in hot water.” - Eleanor Roos

---

### **Step 2 — Add Pagination**

In [5]:
base_url = "https://quotes.toscrape.com/page/{}/"

page = 1

while True:

    url = base_url.format(page)

    response = requests.get(url)

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")

    quotes = soup.find_all("div", class_="quote")

    if not quotes:
        break

    print(f"\nPage {page}\n")

    for q in quotes:
        text = q.find("span", class_="text").text
        author = q.find("small", class_="author").text

        print(text, "-", author)

    page += 1


Page 1

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” - Albert Einstein
“It is our choices, Harry, that show what we truly are, far more than our abilities.” - J.K. Rowling
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” - Albert Einstein
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” - Jane Austen
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.” - Marilyn Monroe
“Try not to become a man of success. Rather become a man of value.” - Albert Einstein
“It is better to be hated for what you are than to be loved for what you are not.” - André Gide
“I have not failed. I've just found 10,000 ways that won't work.” - Thomas A. Edison
“A woman is like a tea bag; you never know how strong it is until it's in hot water.” - Ele

---

### **Step 3 — Add Throttling**

In [6]:
import time
import random

delay = random.uniform(1,3)
time.sleep(delay)

In [9]:
base_url = "https://quotes.toscrape.com/page/{}/"

page = 1

while True:

    url = base_url.format(page)
    
    response = requests.get(url)

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")

    quotes = soup.find_all("div", class_="quote")

    if not quotes:
        break

    print(f"\nPage {page}\n")

    for q in quotes:
        
        
        text = q.find("span", class_="text").text
        author = q.find("small", class_="author").texta
        print(text, "-", author)
    
    
    delay = random.uniform(1,3)
    time.sleep(delay)                                   # random delay added
    print(f"Wait {delay} seconds . . .")
    
    page += 1


Page 1

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” - Albert Einstein
“It is our choices, Harry, that show what we truly are, far more than our abilities.” - J.K. Rowling
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” - Albert Einstein
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” - Jane Austen
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.” - Marilyn Monroe
“Try not to become a man of success. Rather become a man of value.” - Albert Einstein
“It is better to be hated for what you are than to be loved for what you are not.” - André Gide
“I have not failed. I've just found 10,000 ways that won't work.” - Thomas A. Edison
“A woman is like a tea bag; you never know how strong it is until it's in hot water.” - Ele

---

### **Step 4 — Add Retry + Backoff**

In [10]:
def fetch(url):

    delay = 1

    for attempt in range(3):

        try:
            response = requests.get(url, timeout=5)

            if response.status_code == 200:
                return response

        except:
            pass

        print("Retrying in", delay, "seconds")
        time.sleep(delay)
        delay *= 2

    return None

In [14]:
base_url = "https://quotes.toscrape.com/page/{}/"

page = 1

while True:

    url = base_url.format(page)

    response = fetch(url)               # this replace --- response = requests.get(url)


    if not response:
        print("Skipping page")
        break

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")

    quotes = soup.find_all("div", class_="quote")

    if not quotes:
        break

    print(f"\nPage {page}\n")

    for q in quotes:
        text = q.find("span", class_="text").text
        author = q.find("small", class_="author").text

        print(text, "-", author)

    page += 1


Page 1

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” - Albert Einstein
“It is our choices, Harry, that show what we truly are, far more than our abilities.” - J.K. Rowling
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” - Albert Einstein
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” - Jane Austen
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.” - Marilyn Monroe
“Try not to become a man of success. Rather become a man of value.” - Albert Einstein
“It is better to be hated for what you are than to be loved for what you are not.” - André Gide
“I have not failed. I've just found 10,000 ways that won't work.” - Thomas A. Edison
“A woman is like a tea bag; you never know how strong it is until it's in hot water.” - Ele

---

### **Step 5 — Store Data (Structured Output)**

In [12]:
all_data = []

for q in quotes:

    text = q.find("span", class_="text").text
    author = q.find("small", class_="author").text
    tags = [t.text for t in q.find_all("a", class_="tag")]

    all_data.append({
        "text": text,
        "author": author,
        "tags": tags
    })

In [15]:
base_url = "https://quotes.toscrape.com/page/{}/"

page = 1

while True:

    url = base_url.format(page)

    response = fetch(url)              


    if not response:
        print("Skipping page")
        break

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")

    quotes = soup.find_all("div", class_="quote")

    if not quotes:
        break

    print(f"\nPage {page}\n")

    all_data = []                                                       # added here

    for q in quotes:

        text = q.find("span", class_="text").text                           
        author = q.find("small", class_="author").text
        tags = [t.text for t in q.find_all("a", class_="tag")]

        all_data.append({
            "text": text,
            "author": author,
            "tags": tags
        })

        print(text, "-", author)

    page += 1


Page 1

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” - Albert Einstein
“It is our choices, Harry, that show what we truly are, far more than our abilities.” - J.K. Rowling
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” - Albert Einstein
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” - Jane Austen
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.” - Marilyn Monroe
“Try not to become a man of success. Rather become a man of value.” - Albert Einstein
“It is better to be hated for what you are than to be loved for what you are not.” - André Gide
“I have not failed. I've just found 10,000 ways that won't work.” - Thomas A. Edison
“A woman is like a tea bag; you never know how strong it is until it's in hot water.” - Ele

In [16]:
all_data

[{'text': '“The truth." Dumbledore sighed. "It is a beautiful and terrible thing, and should therefore be treated with great caution.”',
  'author': 'J.K. Rowling',
  'tags': ['truth']},
 {'text': "“I'm the one that's got to die when it's time for me to die, so let me live my life the way I want to.”",
  'author': 'Jimi Hendrix',
  'tags': ['death', 'life']},
 {'text': '“To die will be an awfully big adventure.”',
  'author': 'J.M. Barrie',
  'tags': ['adventure', 'love']},
 {'text': '“It takes courage to grow up and become who you really are.”',
  'author': 'E.E. Cummings',
  'tags': ['courage']},
 {'text': '“But better to get hurt by the truth than comforted with a lie.”',
  'author': 'Khaled Hosseini',
  'tags': ['life']},
 {'text': '“You never really understand a person until you consider things from his point of view... Until you climb inside of his skin and walk around in it.”',
  'author': 'Harper Lee',
  'tags': ['better-life-empathy']},
 {'text': '“You have to write the book t

---

### **Step 6 — Save to JSON**

In [17]:
import json

with open("quotes.json", "w", encoding="utf-8") as f:
    json.dump(all_data, f, indent=2)